# AHR PAS-B Screening — Approved-Drug Repurposing

Docking protocol validated by redocking (best pose RMSD **0.49 Å** vs the
crystal indirubin in 7ZUB chain D) — see `research/AHR_redocking_result.md`.

Library: **589 approved drugs**, a MaxMin diversity pick from 2,838 dockable
approved small molecules in ChEMBL (`max_phase=4`), protonated at pH 7.4 and
3D-embedded with RDKit ETKDGv3. Fetched at runtime from a secret GitHub
gist, so this notebook stays small.

**Run on GPU:** Runtime → T4 GPU.

In [ ]:
# 1. GNINA + dependencies
!wget -q https://github.com/gnina/gnina/releases/download/v1.3.3/gnina.cuda12.8.static -O gnina
!chmod +x gnina
!apt-get -qq update && apt-get -qq install -y openbabel 2>/dev/null | tail -1
!pip install -q rdkit biopython pandas
!./gnina --version

In [ ]:
# 2. Receptor prep — identical to the validated redocking
!wget -q https://files.rcsb.org/download/7ZUB.pdb

from Bio.PDB import PDBParser, PDBIO, Select
s = PDBParser(QUIET=True).get_structure('7ZUB', '7ZUB.pdb')
class ProtOnly(Select):
    def accept_chain(self, c): return c.id == 'D'
    def accept_residue(self, r): return r.id[0] == ' '
class LigOnly(Select):
    def accept_residue(self, r): return r.get_resname() == 'JY6'
io = PDBIO(); io.set_structure(s)
io.save('receptor.pdb', ProtOnly())
io.save('indirubin.pdb', LigOnly())
!obabel receptor.pdb -xr -h -p 7.4 -O receptor_prep.pdb 2>/dev/null
!echo "receptor_prep: $(grep -c '^ATOM' receptor_prep.pdb) atoms"

In [ ]:
# 3. Fetch the screening library (secret gist; ChEMBL approved drugs)
!wget -q https://gist.githubusercontent.com/skadlem/3b89119ebd49a7cfa0dd5ad6d64ea903/raw/library_batch1.sdf -O library.sdf
from rdkit import Chem
n = sum(1 for _ in Chem.SDMolSupplier('library.sdf', removeHs=True, sanitize=False))
print(f'library.sdf: {n} compounds')

In [ ]:
# 4. SCREEN — CNN-fast scoring for throughput
!./gnina -r receptor_prep.pdb -l library.sdf \
         --autobox_ligand indirubin.pdb --autobox_add 4 \
         --cnn fast --exhaustiveness 8 --num_modes 3 \
         -o screen.sdf.gz

In [ ]:
# 5. Rank hits and check pocket contacts
from rdkit import Chem
from Bio.PDB import PDBParser
import numpy as np, pandas as pd

# Key residues from research/AHR_pocket.md (His337 closest; Gln383/Tyr322 = H-bonds)
KEY = {'HIS337', 'GLN383', 'TYR322', 'LEU308', 'ILE325', 'LEU353', 'PHE351', 'PHE287'}

rec = PDBParser(QUIET=True).get_structure('R', 'receptor_prep.pdb')[0]
res_atoms = [(res.get_resname() + str(res.id[1]),
              np.array([a.coord for a in res.get_atoms()])) for res in rec.get_residues()]

best_per = {}
for m in Chem.SDMolSupplier('screen.sdf.gz', removeHs=True, sanitize=False):
    if m is None:
        continue
    cid = m.GetProp('_Name')
    px = m.GetConformer().GetPositions()
    contacts = []
    for name, xyz in res_atoms:
        d = np.linalg.norm(xyz - px, axis=1).min()
        if d <= 4.5:
            contacts.append((name, round(float(d), 1)))
    contacts.sort(key=lambda t: t[1])
    rec_info = {
        'id': cid,
        'name': m.GetProp('name') if m.HasProp('name') else '',
        'cnn_aff': float(m.GetProp('CNNaffinity')),
        'cnn_pose': float(m.GetProp('CNNscore')),
        'vina': float(m.GetProp('Affinity')),
        'n_contacts': len(contacts),
        'contacts': contacts,
    }
    if cid not in best_per or rec_info['cnn_aff'] > best_per[cid]['cnn_aff']:
        best_per[cid] = rec_info

ranked = sorted(best_per.values(), key=lambda r: -r['cnn_aff'])
print(f'{len(ranked)} compounds docked\n')
hdr = f"{'CHEMBL':<10}{'name':<22}{'CNNaff':>7}{'CNNpose':>8}{'vina':>7}{'#cont':>6}  key-contacts"
print(hdr); print('-' * len(hdr))
for r in ranked[:20]:
    kc = [n for n, d in r['contacts'] if n in KEY]
    print(f"{r['id']:<10}{r['name'][:21]:<22}{r['cnn_aff']:>7.2f}{r['cnn_pose']:>8.3f}"
          f"{r['vina']:>7.2f}{r['n_contacts']:>6}  {' '.join(kc[:6])}")

df = pd.DataFrame([{k: v for k, v in r.items() if k != 'contacts'} for r in ranked])
df['top_contacts'] = [' '.join(f'{n}({d})' for n, d in r['contacts'][:8]) for r in ranked]
df.to_csv('screen_results.csv', index=False)
print('\nwrote screen_results.csv')
